# Nemotron 3.5 + NeMo — servidor STT live (español, 320 ms)

Tercer backend STT de la tesis, junto a faster-whisper y SimulStreaming. Reemplaza
**solamente** el motor de inferencia en Colab: la captura de audio de la placa, el
bridge WebSocket, el protocolo de sesión, los ACK del firmware y el overlay HDMI se
reutilizan sin cambios.

**Cómo usarla:** `Runtime -> Run all`, esperar a que la celda 12 imprima
`HEALTH: ready` y las URLs, y recién entonces, desde WSL:

```bash
./scripts/audiotestnemotron.sh
```

- Modelo: `nvidia/nemotron-3.5-asr-streaming-0.6b` (público en Hugging Face)
- Runtime: NVIDIA NeMo Speech, **fijado** al commit `2639d4bef8d1450782263a8f616242acfb6fecb9`
- Motor: `nemotron_3_5_nemo` (RNNT cache-aware), **no** faster-whisper ni SimulStreaming
- Idioma: `es-ES` (prompt oficial por request; el default de NeMo es `en-US`)
- Contexto: `[56,3]` = 320 ms de **lookahead algorítmico del modelo**, NO la latencia
  end-to-end hasta el HDMI
- Fin de frase: endpointer oficial `RNNTGreedyEndpointing` (`stop_history_eou`).
  Los eventos distinguen `final_reason=model_eou` de `display_rollup` y
  `session_flush`; sólo el primero es un EOU decidido por el modelo.
- `/health` sólo pasa a `ready` después de cargar el checkpoint, moverlo a CUDA, fijar
  `es-ES` y `[56,3]`, y obtener texto con un canario hablado mediante el mismo camino live.
  ngrok se levanta **después** de eso.

## 1. Verificar GPU

In [ ]:
import os, platform, subprocess, sys
from pathlib import Path
import torch
print('python :', platform.python_version())
print('torch  :', torch.__version__)
print('cuda   :', torch.version.cuda)
print('gpu    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Activá Runtime -> Change runtime type -> GPU antes de continuar'
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], check=True)

## 2. Montar Drive y configurar caches

Mismas rutas que el probe: el checkpoint se descarga una sola vez y se reutiliza desde Drive. No hay que subir nada a mano.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cambiá solamente estas rutas si tu Drive usa otro nombre.
DRIVE_BASE = Path('/content/drive/MyDrive/Tesis-subtitles')
AUDIO_DIR = None            # None => se buscan las ubicaciones ya usadas
PORT = 8765

NEMOTRON_DRIVE = DRIVE_BASE / 'nemotron'
RESULTS_ROOT = NEMOTRON_DRIVE / 'results'
CACHE_ROOT = NEMOTRON_DRIVE / 'cache'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# Los pesos se descargan una vez y se reutilizan desde Drive (~2.4 GB).
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
os.environ['NEMO_CACHE_DIR'] = str(CACHE_ROOT / 'nemo')
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')

def colab_secret(name):
    """os.environ primero, después Colab Secrets. Nunca imprime el valor."""
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

# El modelo es público: la ausencia de HF_TOKEN no bloquea nada, sólo evita
# límites de descarga si existe. Nunca se imprime ni se guarda en resultados.
hf_token = colab_secret('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
print('HF auth:', 'configured' if hf_token else 'anonymous (public model)')
print('results:', RESULTS_ROOT)
print('cache  :', CACHE_ROOT)

## 3. Traer `dev/nemotron`

La rama debe estar pusheada a GitHub. Si esta celda falla con "no existe", pushearla
primero desde WSL: `git push -u origin dev/nemotron`.

In [ ]:
REPO_URL = 'https://github.com/Nacholazabal/subtitle_overlay_fw.git'
REPO_BRANCH = 'dev/nemotron'
REPO_DIR = Path('/content/subtitle_overlay_fw')

remote = subprocess.run(['git', 'ls-remote', '--heads', REPO_URL, REPO_BRANCH], capture_output=True, text=True)
assert remote.stdout.strip(), f'No existe {REPO_BRANCH} en GitHub. Desde WSL: git push -u origin {REPO_BRANCH}'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)

PROJECT_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('project commit:', PROJECT_COMMIT)

## 4. Comprobar físicamente los archivos requeridos

Si esto falla, la copia de Colab está desactualizada; cualquier error posterior sería un síntoma, no la causa.

In [ ]:
REQUIRED = [
    'scripts/__init__.py',
    'scripts/stt_nemotron_backend.py',
    'scripts/stt_nemotron_server.py',
    'scripts/stt_nemotron_probe.py',
    'scripts/stt_simulstreaming_backend.py',
    'scripts/stt_stream_protocol.py',
]
missing = [name for name in REQUIRED if not (REPO_DIR / name).is_file()]
assert not missing, (
    f'La copia de Colab no contiene {missing}. Commit actual: {PROJECT_COMMIT}. '
    'Pusheá dev/nemotron y reejecutá la celda 3.'
)
print('archivos requeridos presentes en', REPO_DIR)

## 5. Limpiar colisiones del paquete `scripts`

Colab puede conservar un paquete ajeno (o una versión anterior) llamado `scripts` en
`sys.modules`. Poner el repo primero en `sys.path` no alcanza si ya está cacheado.

In [ ]:
import importlib

repo_path = str(REPO_DIR)
sys.path[:] = [path for path in sys.path if path != repo_path]
sys.path.insert(0, repo_path)
for module_name in [n for n in list(sys.modules) if n == 'scripts' or n.startswith('scripts.')]:
    del sys.modules[module_name]
importlib.invalidate_caches()

import scripts as project_scripts
expected = (REPO_DIR / 'scripts' / '__init__.py').resolve()
assert Path(project_scripts.__file__).resolve() == expected, (
    f'Se importó otro paquete scripts desde {project_scripts.__file__}'
)
from scripts.stt_nemotron_backend import MODEL_ID, NEMO_COMMIT, NEMO_REPO, RUN_ENGINE
print('project package:', project_scripts.__file__)
print('run engine     :', RUN_ENGINE)
print('model          :', MODEL_ID)
print('NeMo pin       :', NEMO_COMMIT)

## 6. Clonar NeMo en el commit fijado

`2639d4bef8d1450782263a8f616242acfb6fecb9` es el commit con el que se validó el probe.
No se instala `main` flotante.

In [ ]:
NEMO_DIR = Path('/content/NeMo')
if not NEMO_DIR.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', NEMO_REPO, str(NEMO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(NEMO_DIR), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(NEMO_DIR), 'checkout', '-q', NEMO_COMMIT], check=True)
resolved = subprocess.check_output(['git', '-C', str(NEMO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert resolved == NEMO_COMMIT, f'NeMo no quedó fijado: {resolved} != {NEMO_COMMIT}'
subprocess.run(['git', '-C', str(NEMO_DIR), '--no-pager', 'log', '-1', '--oneline'], check=True)
print('NeMo pinned at', resolved)

## 7. Instalar dependencias

No se reemplaza el PyTorch CUDA que trae Colab: sólo se agrega NeMo `[asr]` y las dependencias del servidor.

In [ ]:
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'libsndfile1'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'Cython', 'packaging'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{NEMO_DIR}[asr]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'fastapi', 'uvicorn[standard]', 'pyngrok', 'websockets', 'requests'], check=True)
print('deps installed')

## 8. Comprobar `import nemo.collections.asr` ahora

`pip install -e` registra el checkout con un `.pth` que normalmente se lee recién al
iniciar Python. Este kernel ya estaba vivo, así que agregamos el checkout
explícitamente y comprobamos el import **acá**, no varias celdas después.

In [ ]:
assert (NEMO_DIR / 'nemo').is_dir(), f'el checkout no contiene el paquete nemo: {NEMO_DIR}'
if str(NEMO_DIR) not in sys.path:
    sys.path.insert(0, str(NEMO_DIR))
importlib.invalidate_caches()
import nemo
import nemo.collections.asr as _nemo_asr_smoke
import importlib.metadata
NEMO_TOOLKIT_VERSION = importlib.metadata.version('nemo_toolkit')
print('nemo module   :', nemo.__file__)
print('nemo_toolkit  :', NEMO_TOOLKIT_VERSION)

# API oficial que usa el backend live (falla acá y no a mitad de una sesión).
from nemo.collections.asr.inference.factory.pipeline_builder import PipelineBuilder
from nemo.collections.asr.inference.streaming.framing.request import Frame
from nemo.collections.asr.inference.streaming.framing.request_options import ASRRequestOptions
print('official streaming inference API available')

## 9. Cargar el modelo y hacer un warmup REAL

Carga y warmup ocurren acá, de forma síncrona. Se usan los primeros 12 s de
`desay-short.webm` en Drive para exigir al menos un evento del pipeline live; el silencio
por sí solo no demuestra que el decoder esté aplicando el prompt. Si falla, se muestra la
excepción completa y **no** se levanta el
servidor (nada de "connection refused" infinito).

La primera ejecución descarga ~2.4 GB y tarda varios minutos; las siguientes reutilizan
el cache de Drive.

In [ ]:
import json, time
from scripts.stt_nemotron_server import ServerConfig, BackendState, create_app
from scripts.stt_nemotron_backend import NemotronConfig
from scripts.stt_nemotron_probe import select_drive_audio_dir

canary_audio_dir = select_drive_audio_dir(
    ([Path(AUDIO_DIR)] if AUDIO_DIR else []) + [
        DRIVE_BASE / 'stt-bench' / 'audio',
        DRIVE_BASE / 'simulstreaming' / 'audio',
        DRIVE_BASE / 'nemotron' / 'audio',
    ]
)
canary_audio_path = canary_audio_dir / 'desay-short.webm'
print('streaming speech canary:', canary_audio_path)

backend = NemotronConfig(
    target_lang='es-ES',
    latency_ms=320,          # lookahead algorítmico del modelo -> att_context_size [56,3]
    stop_history_eou_ms=800, # endpointer oficial de NeMo
    residue_tokens_at_end=2,
    strip_lang_tags=True,
    asr_output_granularity='segment',
    compute_dtype='float32', # T4: mismo ajuste validado por el probe
    use_amp=True,
    device='cuda',
    device_id=0,
)
config = ServerConfig(
    host='0.0.0.0', port=PORT, device='cuda', warmup_sec=1.0,
    warmup_audio_path=str(canary_audio_path), warmup_speech_sec=12.0, backend=backend,
)

state = BackendState(config)
state.run_loader()   # síncrono: carga + CUDA + es-ES + [56,3] + warmup + sesión de prueba
if not state.is_ready():
    print('BACKEND FAILED TO LOAD:\n')
    print(state.health_payload().get('error_detail', ''))
    raise RuntimeError('la carga del modelo falló; corregí el error de arriba antes de servir')

health_local = state.health_payload()
assert health_local['streaming_canary']['speech_canary']
assert health_local['streaming_canary']['events_emitted'] > 0
print('streaming canary:', json.dumps(health_local['streaming_canary'], ensure_ascii=False))
print('backend ready. effective config:')
print(json.dumps(health_local['effective_config'], indent=2, ensure_ascii=False))
print('\nprovenance:')
print(json.dumps(health_local['provenance'], indent=2, ensure_ascii=False))

## 10-12. Crear la app, levantar Uvicorn y esperar `/health` ready

In [ ]:
import threading
import uvicorn
import requests

app = create_app(config, backend_state=state)  # startup no recarga: ya está ready
server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()

deadline = time.time() + 120
health = None
while time.time() < deadline:
    try:
        health = requests.get(f'http://127.0.0.1:{PORT}/health', timeout=2).json()
        if health.get('ready'):
            break
    except Exception:
        pass
    time.sleep(0.5)
assert health and health.get('ready'), f'el server no quedó ready: {health}'
assert health['run_engine'] == RUN_ENGINE, f'motor inesperado: {health["run_engine"]}'
print('HEALTH: ready ->', health['run_engine'], health['model'], health['language'], health['device'])

## 13. Abrir ngrok (recién ahora que la readiness es real)

Mismo secret que las notebooks anteriores: agregá un Colab Secret llamado
`NGROK_AUTHTOKEN` (barra lateral izquierda → ícono de llave), o seteá
`os.environ['NGROK_AUTHTOKEN']` antes de esta celda. El token nunca se imprime.

Se usa el **dominio reservado** que ya está configurado como default en los launchers
(`passage-capacity-wistful.ngrok-free.dev`), así `./scripts/audiotestnemotron.sh`
funciona sin pasar `STT_STREAM_URL`. Para usar otro, seteá el secret `NGROK_DOMAIN`.

In [ ]:
from pyngrok import ngrok

NGROK_AUTHTOKEN = colab_secret('NGROK_AUTHTOKEN')
if not NGROK_AUTHTOKEN:
    raise RuntimeError(
        'Falta NGROK_AUTHTOKEN. En Colab, agregalo en Secrets con nombre '
        'NGROK_AUTHTOKEN, o seteálo con os.environ antes de esta celda.'
    )
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Dominio reservado del proyecto: es el default de scripts/run_stt_colab_nemotron.sh.
NGROK_DOMAIN = colab_secret('NGROK_DOMAIN') or 'passage-capacity-wistful.ngrok-free.dev'
ngrok.kill()  # soltar túneles de una ejecución anterior del mismo runtime
tunnel = ngrok.connect(PORT, bind_tls=True, domain=NGROK_DOMAIN)

http_url = str(tunnel.public_url)
ws_url = http_url.replace('https://', 'wss://') + '/stt/stream'

## 14. URL, health y configuración efectiva

In [ ]:
print('=' * 78)
print('Nemotron 3.5 STT server is LIVE')
print('  engine         :', health['run_engine'])
print('  model          :', health['model'])
print('  language       :', health['language'])
print('  device         :', health['device'])
print('  NeMo commit    :', health['nemo_commit'])
print('  NeMo toolkit   :', health['provenance'].get('nemo_toolkit_version'))
print('  torch / CUDA   :', health['provenance'].get('torch_version'), '/', health['provenance'].get('cuda_version'))
print('  GPU            :', health['provenance'].get('gpu_name'))
print('  model revision :', health['provenance'].get('model_revision'))
print('  att context    :', health['effective_config']['att_context_size'],
      '=', health['effective_config']['lookahead_ms'], 'ms de lookahead algorítmico (NO end-to-end)')
print('  chunk size     :', health['provenance'].get('chunk_size_in_secs'), 's')
print('  model load sec :', health.get('model_load_sec'))
print('  HTTP  URL      :', http_url)
print('  HEALTH URL     :', http_url + '/health')
print('  WEBSOCKET URL  :', ws_url)
print('=' * 78)
print('Desde WSL:')
print('  ./scripts/audiotestnemotron.sh')
print('o, si el dominio cambió:')
print(f'  STT_STREAM_URL="{ws_url}" ./scripts/audiotestnemotron.sh')

## 15. Mantener el servidor vivo

In [ ]:
print('Server corriendo. Dejá esta celda activa. Interrumpila para parar.')
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print('stopping')

## (Opcional) Prueba offline standalone a Drive

Transcribe los tres audios vía `/stt/offline` y guarda un JSON chico en
`TESIS/nemotron/results/`. Queda marcado `offline_proxy`: es la salida del mismo motor
sobre el archivo completo, **no** una referencia humana verificada, así que no es WER real.

In [ ]:
from scripts.stt_nemotron_probe import select_drive_audio_dir, EXPECTED_AUDIO_NAMES

candidates = ([Path(AUDIO_DIR)] if AUDIO_DIR else []) + [
    DRIVE_BASE / 'stt-bench' / 'audio',
    DRIVE_BASE / 'simulstreaming' / 'audio',
    DRIVE_BASE / 'nemotron' / 'audio',
]
audio_dir = select_drive_audio_dir(candidates)
out = {
    'run_engine': health['run_engine'],
    'nemo_commit': health['nemo_commit'],
    'model': health['model'],
    'reference_kind': 'offline_proxy',
    'clips': {},
}
for name in EXPECTED_AUDIO_NAMES:
    data = (audio_dir / name).read_bytes()
    response = requests.post(f'http://127.0.0.1:{PORT}/stt/offline', data=data,
                             headers={'X-Audio-Filename': name}, timeout=600)
    response.raise_for_status()
    out['clips'][name] = response.json()
    print(name, '->', out['clips'][name]['text'][:100])
path = RESULTS_ROOT / f'offline_{int(time.time())}.json'
path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved', path)